In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import re
import gc

# Cartoon

In [ ]:
np.random.seed(12345)
M = 10000
z = np.zeros(M) + 2
z[1500:2200] = 3
z[7600:7800] = 1
fig, ax = plt.subplots(dpi=300, figsize = (3, 2))
coverage = 15
step = 1
ax.scatter(
    x=np.arange(M, step=step), 
    y=((np.random.poisson(z * coverage) + np.random.normal(0, scale=1, size=M))/coverage)[::step], 
    s=2,
    marker='.',
    c='k',
    alpha=0.2,
    linewidths=0
)
# ax.plot(z, color='r')
ax.set_ylim(-1, 5)
ax.set_yticks([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])
ax.set_xticks([])
ax.spines['top'].set_position('center')
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

ax.axvspan(1500, 2200, alpha=0.1, color = 'k')
ax.set_ylabel('Copy number')
ax.set_xlabel('Chromosome position')

# plt.savefig('cartoon_plots/cnv_cartoon.pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

np.random.seed(0)
fig, ax = plt.subplots(dpi=300, figsize = (3*5, 2*5))
c1 = 'orchid'
c2 = 'darkseagreen'
y1 = 210
y2 = 190

# ax.set_ylim(y2-150, y1+400)


# Add a rectangle with rounded corners (set boxstyle="round,pad=0.1")
p_arm1_mut = FancyBboxPatch((10, y1+350), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)
q_arm1_mut = FancyBboxPatch((700+30, y1+350), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)

p_arm2_mut = FancyBboxPatch((10, y1+300), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm2_mut = FancyBboxPatch((700+30, y1+300), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm_LOH_mut = FancyBboxPatch((1000+30, y1+300), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)


p_arm1_norm = FancyBboxPatch((10, y1+500), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)
q_arm1_norm = FancyBboxPatch((700+30, y1+500), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)

p_arm2_norm = FancyBboxPatch((10, y1+450), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm2_norm = FancyBboxPatch((700+30, y1+450), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)

# Add the rectangle to the plot
ax.add_patch(p_arm1_norm)
ax.add_patch(q_arm1_norm)

# Add the rectangle to the plot
ax.add_patch(p_arm2_norm)
ax.add_patch(q_arm2_norm)

# Add the rectangle to the plot
ax.add_patch(p_arm1_mut)
ax.add_patch(q_arm1_mut)

# Add the rectangle to the plot
ax.add_patch(p_arm2_mut)
ax.add_patch(q_arm2_mut)
ax.add_patch(q_arm_LOH_mut)

ax.text(1800, y1+325, '10%', fontdict={'fontsize':28})
ax.text(1800, y1+475, '90%', fontdict={'fontsize':28})


het_sites = [100, 850, 930, 1500]
ax.fill_between([0, 2000], y1+10, y2-10, color = 'gray')
# ax.hlines(y2, xmin=0, xmax=2000, color = c2)

het_size = 200

for het in het_sites:
    ax.scatter(het, (y1+y2)/2, s=het_size, marker='*', color='k')
for y in range(2,10+2):
    
    r1_start = 0
    r2_end = 0
    while r2_end < 1500:
        r1_start = r2_end + np.random.geometric(p=0.005)
        r1_end = r1_start + 150
        insert = np.random.poisson(200) 
        r2_start = r1_end + insert
        r2_end = r2_start + 150
        color = 'gray'
        alpha = 0.2
        for het in het_sites: 
            if (r1_start <= het and r1_end >= het) or (r2_start<=het and r2_end>=het):
                color = c1
                alpha = 1
                ax.scatter(x=het, y=y1+(y)*15, marker='*', s=het_size, color=c1, zorder=2, edgecolors='k')
        if r2_end > 2000: continue
        ax.hlines(y1+(y)*15, r1_start, r1_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
        ax.hlines(y1+(y)*15, r1_end, r2_start, linestyles=':', colors=color, zorder=1, alpha=alpha)
        ax.hlines(y1+(y)*15, r2_start, r2_end, linewidth=3, colors=color, zorder=1, alpha=alpha)


for y in range(2, 6+2):
    
    r1_start = 0
    r2_end = 0
    while r2_end < 1500:
        r1_start = r2_end + np.random.geometric(p=0.005)
        r1_end = r1_start + 150
        insert = np.random.poisson(200) 
        r2_start = r1_end + insert
        r2_end = r2_start + 150
        color = 'gray'
        alpha = 0.2
        for het in het_sites: 
            if (r1_start <= het and r1_end >= het) or (r2_start<=het and r2_end>=het):
                color = c2
                alpha = 1
                ax.scatter(x=het, y=y2-(y)*15, marker='*', s=het_size, color=c2, zorder=2, edgecolors='k')

        if r2_end > 2000: continue
        ax.hlines(y2-(y)*15, r1_start, r1_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
        ax.hlines(y2-(y)*15, r1_end, r2_start, linestyles=':', colors=color, zorder=1, alpha=alpha)
        ax.hlines(y2-(y)*15, r2_start, r2_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
            

ax.set_xticks([])
ax.set_yticks([])


# ax.plot([0, 1000], [400, 490], color='k')
# ax.plot([1760, 1900], [490, 400], color='k')
# ax.fill_between([1010, 1750], 475, 750, color='gray', alpha=0.2)
ax.fill_between([0, 1500, 1500, 1520, 1520, 2000], [400, 490, 750, 750, 490, 400], color='gray', alpha=0.2)

for loc in ['left', 'right', 'top', 'bottom']:
    ax.spines[loc].set_visible(False)

# plt.savefig('cartoon_plots/fragments_cartoon.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

np.random.seed(0)
fig, ax = plt.subplots(dpi=300, figsize = (3*5, 2*5))
c1 = 'orchid'
c2 = 'darkseagreen'
y1 = 210
y2 = 190

# ax.set_ylim(y2-150, y1+400)


# Add a rectangle with rounded corners (set boxstyle="round,pad=0.1")
p_arm1_mut = FancyBboxPatch((10, y1+350), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)
q_arm1_mut = FancyBboxPatch((700+30, y1+350), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)

p_arm2_mut = FancyBboxPatch((10, y1+300), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm2_mut = FancyBboxPatch((700+30, y1+300), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm_LOH_mut = FancyBboxPatch((1000+30, y1+300), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)


p_arm1_norm = FancyBboxPatch((10, y1+500), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)
q_arm1_norm = FancyBboxPatch((700+30, y1+500), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c1)

p_arm2_norm = FancyBboxPatch((10, y1+450), 700, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)
q_arm2_norm = FancyBboxPatch((700+30, y1+450), 1000, 20, boxstyle="round,pad=10", edgecolor='black', facecolor=c2)

# Add the rectangle to the plot
ax.add_patch(p_arm1_norm)
ax.add_patch(q_arm1_norm)

# Add the rectangle to the plot
ax.add_patch(p_arm2_norm)
ax.add_patch(q_arm2_norm)

# Add the rectangle to the plot
ax.add_patch(p_arm1_mut)
ax.add_patch(q_arm1_mut)

# Add the rectangle to the plot
# ax.add_patch(p_arm2_mut)
# ax.add_patch(q_arm2_mut)
# ax.add_patch(q_arm_LOH_mut)

ax.text(1800, y1+325, '10%', fontdict={'fontsize':28})
ax.text(1800, y1+475, '90%', fontdict={'fontsize':28})


het_sites = [100, 850, 930, 1500]
ax.fill_between([0, 2000], y1+10, y2-10, color = 'gray')
# ax.hlines(y2, xmin=0, xmax=2000, color = c2)

het_size = 200

for het in het_sites:
    ax.scatter(het, (y1+y2)/2, s=het_size, marker='*', color='k')
for y in range(2,10+2):
    
    r1_start = 0
    r2_end = 0
    while r2_end < 1500:
        r1_start = r2_end + np.random.geometric(p=0.005)
        r1_end = r1_start + 150
        insert = np.random.poisson(200) 
        r2_start = r1_end + insert
        r2_end = r2_start + 150
        color = 'gray'
        alpha = 1
        for het in het_sites: 
            if (r1_start <= het and r1_end >= het) or (r2_start<=het and r2_end>=het):
                color = 'gray'
                alpha = 1
                ax.scatter(x=het, y=y1+(y)*15, marker='*', s=het_size, color=c1, zorder=2, edgecolors='k')
        if r2_end > 2000: continue
        ax.hlines(y1+(y)*15, r1_start, r1_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
        ax.hlines(y1+(y)*15, r1_end, r2_start, linestyles=':', colors=color, zorder=1, alpha=alpha)
        ax.hlines(y1+(y)*15, r2_start, r2_end, linewidth=3, colors=color, zorder=1, alpha=alpha)


for y in range(2, 6+2):
    
    r1_start = 0
    r2_end = 0
    while r2_end < 1500:
        r1_start = r2_end + np.random.geometric(p=0.005)
        r1_end = r1_start + 150
        insert = np.random.poisson(200) 
        r2_start = r1_end + insert
        r2_end = r2_start + 150
        color = 'gray'
        alpha = 1
        for het in het_sites: 
            if (r1_start <= het and r1_end >= het) or (r2_start<=het and r2_end>=het):
                color = 'gray'
                alpha = 1
                ax.scatter(x=het, y=y2-(y)*15, marker='*', s=het_size, color=c2, zorder=2, edgecolors='k')

        if r2_end > 2000: continue
        ax.hlines(y2-(y)*15, r1_start, r1_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
        ax.hlines(y2-(y)*15, r1_end, r2_start, linestyles=':', colors=color, zorder=1, alpha=alpha)
        ax.hlines(y2-(y)*15, r2_start, r2_end, linewidth=3, colors=color, zorder=1, alpha=alpha)
            

ax.set_xticks([])
ax.set_yticks([])


# ax.plot([0, 1000], [400, 490], color='k')
# ax.plot([1760, 1900], [490, 400], color='k')
# ax.fill_between([1010, 1750], 475, 750, color='gray', alpha=0.2)
ax.fill_between([0, 1500, 1500, 1520, 1520, 2000], [400, 490, 750, 750, 490, 400], color='gray', alpha=0.2)

for loc in ['left', 'right', 'top', 'bottom']:
    ax.spines[loc].set_visible(False)

# plt.savefig('cartoon_plots/fragments_cartoon.pdf', format='pdf', transparent=True)
plt.show()

In [ ]:
np.random.seed(12345)
M = 10000
z = np.zeros(M)
baf = 0.05
z[3000:] = baf
z[7900:] = -baf
fig, ax = plt.subplots(dpi=300, figsize = (3, 2))
colors = ['gold' if hidden != 0 else 'k' for hidden in z]

ax.scatter(
    range(M), 
    (np.random.binomial(30, z+0.5) + np.random.normal(0, scale=1, size=M))/30, 
    marker='.', 
    s=5,
    c=colors, 
    edgecolors='k', 
    linewidths=0,
    alpha=0.4
)

ax.set_xticks([])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.set_ylim([0, 1])
ax.spines['top'].set_position('center')
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.plot(z+0.5, color='r', linewidth=2)
ax.set_ylabel('Phased allelic fraction')
ax.set_xlabel('Chromosome position')
# plt.savefig('cartoon_plots/baf_cartoon.pdf', transparent=True, bbox_inches='tight')
plt.show()

# Depth Profile

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(10, 8), sharey=True, sharex=True, dpi=150)
for j in range(2):

    depth = pd.read_csv(f'/mnt/project/lohdata/david/mCAs_WGS/plots/example_depth_profile_{j+1}.depth.txt', sep='\t')
    depth = depth.query('bpStart > 20e6 and bpEnd<70e6')

    depth = depth.assign(bin = lambda x: (x['bpStart']/50e3).astype(int)).groupby('bin').agg(
        start = ('bpStart', 'min'),
        OBSreads = ('OBSreads', 'sum'),
        EXPreads = ('EXPreads', 'sum'),
        EXPreadsNoPCadj = ('EXPreadsNoPCadj', 'sum')
    )
    for i in [0, 2]:
        ax[i][j].plot(
            depth['start']/1e6,
            depth['OBSreads']/depth['EXPreadsNoPCadj'],
            color='plum',
            label = 'No PC-adjustment'
        )
    for i in [1, 2]:
        ax[i][j].plot(
            depth['start']/1e6,
            depth['OBSreads']/depth['EXPreads'],
            color = 'peru',
            label = 'PC-adjusted'
        )
    for i in [0, 1, 2]:
        ax[i][j].axhline(1, color='black', linestyle='--')
    ax[2][j].set_xlabel('Chromosome 1 Position (Mb)', fontsize=18)
    ax[0][j].set_title(f'Individual {j+1}', fontsize=18)

ax[1][0].set_ylabel('Observed Reads/Expected Reads', fontsize=18)
for i in range(3):
    for j in range(2):
        ax[i][j].tick_params(labelsize=12, axis='both')
        ax[i][j].legend(frameon=False)

plt.savefig('example_depth_profiles.pdf', bbox_inches='tight', transparent=True)

# SNP-array comp

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df_snp = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.mCA_calls.snp_array.txt', sep ='\t')
df_age = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID_age.40709.txt', sep='\t')
df_age['ageBin'] = df_age['age']//5 * 5
name_map = {
    'neutral': 'CN-LOH',
    'gain': 'GAIN',
    'loss': 'LOSS',
    'unknown': 'UNDETERMINED'
}
df_snp['TYPE'] = [name_map[x] for x in df_snp['COPY_CHANGE']]
df_snp = df_snp.merge(df_age)

In [ ]:
df_snp.merge(df.assign(CHR=lambda x: [int(y[3:]) for y in x['chr']]), on=['ID', 'CHR'])['ID'].nunique()

In [ ]:

print(df_snp.merge(
    df.assign(CHR=lambda x: [int(y[3:]) for y in x['chr']]), 
    on=['ID', 'CHR'])[['ID', 'CHR']].drop_duplicates().shape[0])
print(df_snp[['ID', 'CHR']].drop_duplicates().shape[0])


In [ ]:
snp_isodisomy_cnt = {}
for chrom,length in df_snp.groupby('CHR').agg(max_len = ('SIZE_MB', 'max')).iterrows():
    if chrom == 13 or chrom == 14 or chrom == 15 or chrom == 21 or chrom == 22: continue
    length = length['max_len']
    snp_isodisomy_cnt[f"chr{chrom}"] = df_snp.query('COPY_CHANGE=="neutral" and CHR == @chrom and SIZE_MB +10 > @length').shape[0]
isodisomy_cnt = df.query('type=="CN-LOH" and p=="T" and q=="T"').groupby('chr').agg(
    wgs_isodisomy_cnt = ('ID', 'nunique')
).reset_index().merge(
    pd.DataFrame.from_dict(snp_isodisomy_cnt, orient='index', columns=['snp_isodisomy_cnt']).reset_index().rename(columns={'index':'chr'})
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
ax.bar(
    x = [int(x[3:]) for x in isodisomy_cnt['chr']],
    height = isodisomy_cnt['wgs_isodisomy_cnt'],
    color = 'darkgoldenrod',
    alpha = 1,
    label = 'WGS',
    edgecolor = 'k',
)
ax.bar(
    x = [int(x[3:]) for x in isodisomy_cnt['chr']],
    height = isodisomy_cnt['snp_isodisomy_cnt'],
    color = 'khaki',
    alpha = 1,
    label = 'SNP array',
    edgecolor = 'k',
)
ax.axvspan(13-0.5, 15+0.5, color='gray', alpha=0.2, hatch='//')
ax.axvspan(21-0.5, 22+0.5, color='gray', alpha=0.2, hatch='//')
ax.set_xticks(range(1, 23), [f'chr{x}' for x in range(1, 23)], rotation=45)
ax.set_ylim(0, 130)
ax.legend(frameon=False, fontsize=14)
ax.set_ylabel('Number of isodisomy events', fontsize=16)
ax.set_xlabel('Chromosome', fontsize=16)


In [ ]:
!pip install -q sankeyflow
from sankeyflow import Sankey

counts = df_snp[['ID', 'CHR', 'SIZE_MB', 'COPY_CHANGE']].merge(
    df.assign(CHR = lambda x: [int(y[3:]) for y in x['chr']])[['ID', 'CHR', 'length', 'type']]
).query('SIZE_MB - length/1e6 < 0.1 * SIZE_MB').assign(
    SNP_COPY_CHANGE = lambda x: [y.upper() + " (SNP-array)" if y != 'neutral' else 'CN-LOH (SNP-array)' for y in x['COPY_CHANGE']],
    WGS_COPY_CHANGE = lambda x: x['type']
).groupby(['SNP_COPY_CHANGE', 'WGS_COPY_CHANGE']).size().reset_index(name='count')

flows = list(counts.to_records(index=False))
alpha = 0.4
cmap = {'CN-LOH': (1.0, 0.84313725, 0.0, alpha), 'LOSS': (0.0, 0.0, 1.0, alpha), 'GAIN': (1.0, 0.0, 0.0, alpha)}
flows = [(x[0], x[1], x[2], {'color': cmap[x[1]]}) for x in flows]


fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
s = Sankey(flows=flows)
for i in [0, 1]:
    for node in s.nodes[i]:
        node.color='gold' if 'CN-LOH' in node.label else 'red' if 'GAIN' in node.label else 'blue' if 'LOSS' in node.label else 'gray'
    
s.draw(ax)
plt.savefig('snp_vs_wgs_mCA_classification.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize = (4*1.5, 3*1.5), dpi=150)
colors = {
    'CN-LOH': 'gold',
    'GAIN': 'red',
    'LOSS': 'blue',
    'UNDETERMINED': 'gray'
}
width = 2
bottom = np.zeros(6)
for key, grp in df \
    .groupby(['ageBin', 'type'])\
    .count()[['ID']] \
    .reset_index() \
    .merge(df_age.query('ageBin>=40 and ageBin<70').groupby('ageBin').count()[['ID']], on = 'ageBin') \
    .pipe(lambda df: df.assign(prevalence=df['ID_x']/df['ID_y'])) \
    .drop(['ID_x', 'ID_y'], axis = 1) \
    .groupby('type'):
    ax.bar(grp['ageBin']+width/2, grp['prevalence'], width=width, color = colors[key], alpha=0.8, label=f'WGS {key}', bottom=bottom)
    bottom+=grp['prevalence'].to_numpy()

bottom-=bottom
for key, grp in df_snp \
    .groupby(['ageBin', 'TYPE'])\
    .count()[['ID']] \
    .reset_index() \
    .merge(df_age.query('ageBin>=40 and ageBin<70').groupby('ageBin').count()[['ID']], on = 'ageBin') \
    .pipe(lambda df: df.assign(prevalence=df['ID_x']/df['ID_y'])) \
    .drop(['ID_x', 'ID_y'], axis = 1) \
    .groupby('TYPE'):
    ax.bar(grp['ageBin']-width/2, grp['prevalence'],  width=width, color = colors[key], alpha = 0.5, hatch='///', label=f'SNP-array {key}', bottom=bottom)
    bottom+=grp['prevalence'].to_numpy()

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc='upper left', frameon=False, ncol=2)
ax.set_xticks(
    [40, 45, 50, 55, 60, 65],
    ['40-44','45-49','50-54','55-59','60-64','65-69']
)
ax.set_xlabel('Age (years)', fontdict={'fontsize':18})
ax.set_ylabel('Autosomal mCAs per indiv.', fontdict={'fontsize':18})
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.savefig('prevalence.pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize = (4*1.5, 3*1.5), dpi=150)

bins = np.logspace(-3, 0, 50)
# ax1.hist(df['cf'].astype(float), bins = bins, alpha=0.5, label='WGS')
# ax1.hist(df_snp.query('CELL_FRAC!="unknown"')['CELL_FRAC'].astype(float), bins = bins, alpha=0.5, label='SNP-array')
ax1.hist(df['bdev'].astype(float), bins = bins, alpha=1, label='WGS', edgecolor='k', color='lightgray')
ax1.hist(df_snp['BAF'].astype(float), bins = bins, alpha=1, label='SNP-array', edgecolor='k', hatch='///', color='gray')
ax1.set_xscale('log')
ax1.set_xlabel('mCA allelic imbalance (|BAF - 0.5|)', fontdict={'fontsize':18})
ax1.set_xticks([0.001, 0.01, 0.1, 1], [0.001, 0.01, 0.1, 1])

bins = np.logspace(-3, 3, 50)
ax2.hist(df['length'].astype(float)/1e6, bins = bins, alpha=1, label='WGS', edgecolor='k', color='lightgray')
ax2.hist(df_snp['SIZE_MB'].astype(float), bins = bins, alpha=1, label='SNP-array', edgecolor='k', hatch='///', color='gray')
ax2.set_xscale('log')
ax2.set_xlabel('mCA size (Mb)', fontdict={'fontsize':18})
ax2.set_xticks([0.001, 0.01, 0.1, 1, 10, 100], [0.001, 0.01, 0.1, 1, 10, 100])
ax1.legend(frameon=False)
fig.supylabel('Number of mCAs', fontsize=18)
plt.tight_layout()
plt.savefig('mCA_size_and_cf.pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
np.random.seed(12345)
def delBDev2DepthDev(bdev):
    return -2 * bdev / (2 * bdev + 1)

def dupBDev2DepthDev(bdev):
    return -2 * bdev / (2 * bdev - 1)
    
cmap = {'GAIN':'r', 'LOSS':'b', 'CN-LOH':'gold'}
fig, ax = plt.subplots(figsize=(4*1.5, 3*1.5), dpi=150)

X = df['bdev'].to_numpy()
Y = df['depth'].to_numpy()
colors = np.array([cmap[cn] for cn in df['type']])

order = np.random.choice(len(df), len(df), replace=False)

# # ax.set_xlim(0, 0.01)
# # ax.set_ylim(-0.02, 0.02)
ax.scatter(X[order], Y[order], color=colors[order], s=0.1, alpha=0.3, rasterized=True)
ax.set_xlabel('mCA allelic imbalance (|BAF - 0.5|)', fontdict={'fontsize':18})
ax.set_ylabel('Relative WGS read-depth', fontdict={'fontsize':18})

ax.plot(np.arange(0, 1/6+0.05, 0.001), dupBDev2DepthDev(np.arange(0, 1/6+0.05, 0.001)), color=cmap['GAIN'])
ax.plot(np.arange(0, 1/2, 0.001), delBDev2DepthDev(np.arange(0, 1/2, 0.001)), color=cmap['LOSS'])
ax.plot([0, 0.5], [0, 0], color=cmap['CN-LOH'])
ax.text(0.1, 0.6, 'Gain', color='red', fontsize=18)
ax.text(0.3, 0.1, 'CN-LOH', color='gold', fontsize=18)
ax.text(0.4, -0.4, 'Loss', color='blue', fontsize=18)
plt.savefig('depth_vs_baf.pdf', transparent=True, bbox_inches='tight')

# Pileups

In [ ]:
from matplotlib.ticker import MultipleLocator
cyto = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/cytoBand.txt.gz', sep ='\t', header=None)
cyto.columns = ['chrom', 'start', 'stop', 'label', 'stain']
cyto['start'] = cyto['start']/1e6
cyto['stop'] = cyto['stop']/1e6
cyto['width'] = cyto['stop'] - cyto['start']

color_lookup = {
    'gneg': (1., 1., 1.),
    'gpos25': (.6, .6, .6),
    'gpos50': (.4, .4, .4),
    'gpos75': (.2, .2, .2),
    'gpos100': (0., 0., 0.),
    'acen': (.8, .4, .4),
    'gvar': (.8, .8, .8),
    'stalk': (.9, .9, .9)
}

def natural_sort_key(s):
    """A natural sort key function."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

for chrom, df_chr in sorted(df.groupby('chr'), key=lambda x: natural_sort_key(x[0])[1]):
    fig, ax = plt.subplots(figsize=(df_chr['bpEnd'].max()/1e7/2,(len(df_chr)+100)/1e2/2))
    sample, counts = np.unique(df_chr.ID, return_counts=True)
    recurrent_samples = {x: 1 if y > 1 else 0 for x, y in zip(sample, counts)}
    df_chr['recurrent'] = [recurrent_samples[x] for x in df_chr.ID]
    df_chr['y'] = np.argsort(np.argsort(
        df_chr['bpStart'] - df_chr['length'] + 1e9 * (
            4 * (df_chr['recurrent'] == 1).astype(int) +
            3 * (df_chr['type'] == 'LOSS').astype(int) +
            2 * (df_chr['type'] == 'CN-LOH').astype(int) +
            1 * (df_chr['type'] == 'GAIN').astype(int) + 
            0.1 * (df_chr['p'] == 'N').astype(int)
        ) 
    ))
    y_pos = {x: y for x, y in zip(df_chr['ID'], df_chr['y'])}
    # df_chr['y'] = np.argsort(np.argsort([y_pos[x] for x in df_chr.ID]))
    # df_chr['y'] = [y_pos[x] for x in df_chr.ID]
    df_chr['y'] = np.unique([y_pos[x] for x in df_chr.ID], return_inverse=True)[1]
    df_chr['y'] += 50 * df_chr['recurrent']

    cmap = {'CN-LOH': 'gold', 'LOSS': 'blue', 'GAIN': 'red', 'UNDETERMINED':'gray'}
    ax.axhline(
        y=(counts == 1).sum() + 25, 
        color='black'
    )

    ax.hlines(
        xmin=df_chr['bpStart']/1e6, 
        xmax=df_chr['bpEnd']/1e6, 
        y=df_chr['y'], 
        colors=[cmap[x] for x in df_chr['type']],
        linewidth=1,
        rasterized=True,
    )

    cyto_chr = cyto.query('chrom==@chrom')
    ax.broken_barh(
        cyto_chr.loc[:, ['start', 'width']].to_numpy(), 
        [-75, 50], 
        facecolor = [color_lookup[x] for x in cyto_chr['stain']],
        edgecolor = 'black'
    )
    ax.set_yticks([])
    xticks = np.arange(0, df_chr['bpEnd'].max()/1e6, 50)
    xtick_labels = [f'{int(x)}Mb' if x != 0 else '' for x in xticks]
    ax.set_xticks(xticks, xtick_labels, fontsize=24)
    ax.set_xlim(0, df_chr['bpEnd'].max()/1e6+1)
    ax.set_ylim(-100, df_chr['y'].max())
    ax.xaxis.set_major_locator(MultipleLocator(50)) 
    ax.xaxis.set_minor_locator(MultipleLocator(10)) 
    ax.tick_params(which='major', width=1.00, length=10)
    ax.tick_params(which='minor', width=0.75, length=5)

    # ax.set_xlabel("Position (Mb)")
    ax.text(-20, -100, chrom[3:], fontsize=42)
    
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(True)

    plt.savefig(f'{chrom}.pileup.pdf', transparent=True, bbox_inches='tight')
    plt.close()

## High Res

In [ ]:
# from matplotlib.ticker import MultipleLocator
# cyto = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/GRCh38_supp_files/cytoBand.txt.gz', sep ='\t', header=None)
# cyto.columns = ['chrom', 'start', 'stop', 'label', 'stain']
# cyto['start'] = cyto['start']/1e6
# cyto['stop'] = cyto['stop']/1e6
# cyto['width'] = cyto['stop'] - cyto['start']

# color_lookup = {
#     'gneg': (1., 1., 1.),
#     'gpos25': (.6, .6, .6),
#     'gpos50': (.4, .4, .4),
#     'gpos75': (.2, .2, .2),
#     'gpos100': (0., 0., 0.),
#     'acen': (.8, .4, .4),
#     'gvar': (.8, .8, .8),
#     'stalk': (.9, .9, .9)
# }

# def natural_sort_key(s):
#     """A natural sort key function."""
#     return [int(text) if text.isdigit() else text.lower()
#             for text in re.split(r'(\d+)', s)]

# for chrom, df_chr in sorted(df.groupby('chr'), key=lambda x: natural_sort_key(x[0])[1]):
#     fig, ax = plt.subplots(figsize=(10,12))
#     sample, counts = np.unique(df_chr.ID, return_counts=True)
#     recurrent_samples = {x: 1 if y > 1 else 0 for x, y in zip(sample, counts)}
#     df_chr['recurrent'] = [recurrent_samples[x] for x in df_chr.ID]
#     df_chr['y'] = np.argsort(np.argsort(
#         df_chr['bpStart'] - df_chr['length'] + 1e9 * (
#             4 * (df_chr['recurrent'] == 1).astype(int) +
#             3 * (df_chr['type'] == 'LOSS').astype(int) +
#             2 * (df_chr['type'] == 'CN-LOH').astype(int) +
#             1 * (df_chr['type'] == 'GAIN').astype(int) + 
#             0.1 * (df_chr['p'] == 'N').astype(int)
#         ) 
#     ))
#     y_pos = {x: y for x, y in zip(df_chr['ID'], df_chr['y'])}
#     # df_chr['y'] = np.argsort(np.argsort([y_pos[x] for x in df_chr.ID]))
#     # df_chr['y'] = [y_pos[x] for x in df_chr.ID]
#     df_chr['y'] = np.unique([y_pos[x] for x in df_chr.ID], return_inverse=True)[1]
#     df_chr['y'] += 50 * df_chr['recurrent']

#     cmap = {'CN-LOH': 'gold', 'LOSS': 'blue', 'GAIN': 'red', 'UNDETERMINED':'gray'}
#     ax.axhline(
#         y=(counts == 1).sum() + 25, 
#         color='black'
#     )

#     ax.hlines(
#         xmin=df_chr['bpStart']/1e6, 
#         xmax=df_chr['bpEnd']/1e6, 
#         y=df_chr['y'], 
#         colors=[cmap[x] for x in df_chr['type']],
#         linewidth=1,
#         rasterized=False,
#     )

#     cyto_chr = cyto.query('chrom==@chrom')
#     ax.broken_barh(
#         cyto_chr.loc[:, ['start', 'width']].to_numpy()/cyto_chr['stop'].max(), 
#         [0.01, 0.04], 
#         facecolor = [color_lookup[x] for x in cyto_chr['stain']],
#         edgecolor = 'black',
#         transform=ax.transAxes
#     )
#     ax.set_yticks([])
#     xticks = np.arange(0, df_chr['bpEnd'].max()/1e6, 50)
#     xtick_labels = [int(x) for x in xticks]
#     ax.set_xticks(xticks, xtick_labels, fontsize=12, rotation=0)
#     ax.xaxis.tick_top()
#     ax.set_xlim(0, df_chr['bpEnd'].max()/1e6+1)
#     ax.set_ylim(-df_chr['y'].max()/20, df_chr['y'].max())
#     ax.xaxis.set_major_locator(MultipleLocator(50)) 
#     ax.xaxis.set_minor_locator(MultipleLocator(10)) 
#     ax.tick_params(which='major', width=1.00, length=10)
#     ax.tick_params(which='minor', width=0.75, length=5)
#     ax.grid(axis='x', which='major', linestyle='--', alpha=0.5)

#     # ax.set_xlabel(f"{chrom} position (Mb)", fontsize=24)
#     # ax.text(-0.02, 0, chrom[3:], fontsize=42, transform=ax.transAxes, ha='right')
    
#     ax.spines['left'].set_visible(False)
#     ax.spines['right'].set_visible(False)
#     ax.spines['top'].set_visible(True)
#     ax.spines['bottom'].set_visible(False)
#     for _,row in cyto_chr.iterrows():
#         ax.annotate(
#             row['label'],
#             xy=(((row['start'] + row['stop'])/2)/cyto_chr['stop'].max(), 0.01), 
#             xytext=(((row['start'] + row['stop'])/2)/cyto_chr['stop'].max(), 0), 
#             xycoords="axes fraction",
#             textcoords="axes fraction",
#             fontsize=8, 
#             rotation=90, 
#             ha='center', 
#             va='top',
#         )
#     plt.savefig(f'{chrom}.high_res.pileup.pdf', transparent=True, bbox_inches='tight')
#     plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(1, 1), dpi=300)

ax.plot([],[], color='blue', label='Loss')
ax.plot([],[], color='gold', label='CN-LOH')
ax.plot([],[], color='red', label='Gain')
ax.legend(loc='center', frameon=False)
ax.axis('off')
plt.savefig('legend.pdf', transparent=True, bbox_inches='tight')

# Supp

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.validation.txt.gz', sep='\t')
df['bins'] = (np.floor((df['expectedValRate']-1e-6) * 100/5) * 5).astype(int)
df['concordant'] = (df['zscore']> 0)
df['discordant'] = (df['zscore']< 0)
df = df.query('bins>0').groupby('bins').agg(
    concordant = ('concordant', sum),
    discordant = ('discordant', sum),
    expected = ('expectedValRate', np.mean)
).reset_index()
df['rate'] = df['concordant']/(df['concordant']+ df['discordant'])

df['lower'] = scipy.stats.beta(df['concordant']+1/2, df['discordant']+1/2).ppf(0.025)
df['upper'] = scipy.stats.beta(df['concordant']+1/2, df['discordant']+1/2).ppf(0.975)

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
ax.errorbar(df['expected'], df['rate'], yerr=(df['rate']-df['lower'], df['upper']-df['rate']), capsize=5)
ax.scatter(df['expected'], df['rate'])
ax.plot([0.2, 1], [0.2, 1], color='r')
ax.set_xlim(0.2, 1)
ax.set_ylim(0.2, 1)
ax.set_xlabel('Expected WES validation rate', fontsize=18)
ax.set_ylabel('Observed WES validation rate', fontsize=18)
plt.savefig('validation_rate.pdf', transparent=True, bbox_inches='tight')

In [ ]:
np.random.seed(12345)

cmap = {'GAIN':'r', 'LOSS':'b', 'CN-LOH':'gold'}


df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
# df = df.query('bdev<0.02')

X = df['bdev'].to_numpy()
Y = df['depth'].to_numpy()
colors = np.array([cmap[cn] for cn in df['type']])

order = np.random.choice(len(df), len(df), replace=False)

fig, ax = plt.subplots(2, 3, figsize=(16, 10), dpi=150)
labels = [['a', 'c', 'e'], ['b', 'd', 'f']]

for i in range(2):
    for j in range(3):
        alpha = 0.3 if j==0 else 0.1
        ax[i][j].scatter(X[order], Y[order], color=colors[order] if i==1 else 'gray', s=0.5, alpha=alpha, rasterized=True)
        ax[i][j].text(-0.03, 1.05, labels[i][j], transform = ax[i][j].transAxes, fontsize=24, ha='right')


for j in range(3):
    ax[1][j].axline((0, 0), slope=0.8, color='k', linestyle='--')
    ax[1][j].axline((0, 0), slope=-0.6, color='k', linestyle='--')
    ax[1][j].set_xlabel('|BAF - 0.5|', fontdict={'fontsize':18})

for i in range(2):
    ax[i][0].set_xlim(0, 0.01)
    ax[i][0].set_ylim(-0.01, 0.01)
    ax[i][0].set_ylabel('Centered relative\nWGS read-depth', fontdict={'fontsize':18})
    ax[i][1].set_xlim(0, 0.05)
    ax[i][1].set_ylim(-0.05, 0.05)
    ax[i][2].set_xlim(0, 0.3)
    ax[i][2].set_ylim(-0.3, 0.3)


plt.savefig('depth_vs_baf_zoomins.pdf', transparent=True, bbox_inches='tight')

In [ ]:
np.random.seed(12345)

cmap = {'GAIN':'r', 'LOSS':'b', 'CN-LOH':'gold'}


df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
# df = df.query('bdev<0.02')


labels = [['a', 'b'], ['c', 'd']]
conditions = [['length<2e6', 'length>2e6 and length<10e6'] , ['length>10e6 and length<50e6', 'length>50e6']]
titles = ['0-2Mb', '2-10Mb', '10-50Mb', '50-250Mb']
fig, ax = plt.subplots(2, 2, figsize=(12, 12), dpi=150, sharex=True, sharey=True)
for i, row in enumerate(conditions):
    for j, condition in enumerate(row):
        alpha = 0.3
        X = df.query(condition)['bdev'].to_numpy()
        Y = df.query(condition)['depth'].to_numpy()
        order = np.random.choice(len(X), len(X), replace=False)
        colors = np.array([cmap[cn] for cn in df.query(condition)['type']])
        ax[i][j].scatter(X[order], Y[order], color=colors[order], s=1, alpha=alpha, rasterized=True)
        ax[i][j].text(-0.05, 1.05, labels[i][j], transform = ax[i][j].transAxes, fontsize=24, ha='right')
        ax[i][j].set_title(f'mCAs of length {titles[i*2+j]}', fontsize=18)
        ax[i][j].plot(np.arange(0, 1/6+0.05, 0.001), dupBDev2DepthDev(np.arange(0, 1/6+0.05, 0.001)), color=cmap['GAIN'], alpha=0.3)
        ax[i][j].plot(np.arange(0, 0.35, 0.001), delBDev2DepthDev(np.arange(0, 0.35, 0.001)), color=cmap['LOSS'], alpha=0.3)
        ax[i][j].tick_params(labelsize=14)
        ax[i][j].text(0., 0.6, f'n={len(X)}', color='k', fontsize=18)
        if i==1: ax[i][j].set_xlabel('|BAF - 0.5|', fontsize=18)
        if j==0: ax[i][j].set_ylabel('Relative WGS read-depth', fontsize=18)
    plt.savefig('depth_vs_baf_by_size.pdf', transparent=True, bbox_inches='tight')

In [ ]:
np.random.seed(12345)

cmap = {'GAIN':'r', 'LOSS':'b', 'CN-LOH':'gold'}


df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df = df.query('length>10e6')

X = df['bdev'].to_numpy()
Y = df['depth'].to_numpy()
colors = np.array([cmap[cn] for cn in df['type']])

order = np.random.choice(len(df), len(df), replace=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=150, sharex=True)

alpha = 1
ax[0].scatter(X[order], Y[order], color=colors[order] if i==1 else 'gray', s=0.1, alpha=alpha, rasterized=True)
ax[1].scatter(X[order], np.log(1+Y[order]), color=colors[order] if i==1 else 'gray', s=0.1, alpha=alpha, rasterized=True)
ax[0].set_ylim(-0.7, 0.8)
ax[1].set_ylim(-0.7, 0.8)
ax[0].set_xlabel('|BAF - 0.5|', fontsize=14)
ax[1].set_xlabel('|BAF - 0.5|', fontsize=14)
ax[0].set_ylabel('Relative depth', fontsize=14)
ax[1].set_ylabel('log(1 + Relative depth)', fontsize=14)
plt.tight_layout()
ax[0].plot(np.arange(0, 0.2, 0.001), dupBDev2DepthDev(np.arange(0, 0.2, 0.001)), color='r')
ax[0].plot(np.arange(0, 0.3, 0.001), delBDev2DepthDev(np.arange(0, 0.3, 0.001)), color='b')
ax[1].plot(np.arange(0, 0.2, 0.001), np.log(1+dupBDev2DepthDev(np.arange(0, 0.2, 0.001))), color='r')
ax[1].plot(np.arange(0, 0.3, 0.001), np.log(1+delBDev2DepthDev(np.arange(0, 0.3, 0.001))), color='b')
plt.show()

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df.groupby('chr').agg(
    CNLOH = ('type', lambda x: (x == 'CN-LOH').sum()),
    LOSS = ('type', lambda x: (x == 'LOSS').sum()),   
    GAIN = ('type', lambda x: (x == 'GAIN').sum()),   
    Total = ('ID', 'count'),
).reset_index().rename(columns={'chr':'Chromosome'}).to_csv('per_chr_mCA_counts.csv', index=False)

In [ ]:
df = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/calls/WGS_500k.calls.txt', sep ='\t')
df_cancer = pd.read_csv('/mnt/project/lohdata/david/mCAs_WGS/sample_data/ID.histology.behavior.sample_date.cancer_date.txt', sep='\t')
df_cancer = df_cancer \
    .query('cancer_date < collection_date') \
    .groupby('ID') \
    .agg(PREV_CANCER=('ID', lambda x: 1 if x.count() > 0 else 0)).reset_index()
df \
    .merge(df_cancer, on='ID', how='left').fillna(0).astype({'PREV_CANCER':'int'}) \
    .assign(chrNum=lambda x: [int(y[3:]) for y in x['chr']]) \
    .sort_values(['chrNum', 'bpStart']) \
    .assign(ID=lambda x: pd.factorize(x['ID'])[0]+1) \
    .reset_index(drop=True) \
    .assign(
        AGE=lambda x: [f"{lower}-{lower+4}" for lower in x['ageBin']],
        START_MB=lambda x: (x['bpStart']/1e6).round(3),
        END_MB=lambda x: (x['bpEnd']/1e6).round(3),
        SIZE_MB=lambda x: (x['length']/1e6).round(3),
    ) \
    .rename(columns={
        'chr': 'CHR',
        'type': 'COPY_CHANGE',
        'bdev': 'BAF',
        'bdevSE': 'BAF_SE',
        'depth': 'DEPTH',
        'depthSE': 'DEPTH_SE',
        'cf': 'CELL_FRAC',
        'sex': 'SEX'
    }) \
    [['ID', 'SEX', 'AGE', 'PREV_CANCER', 'CHR', 'START_MB', 'END_MB', 'SIZE_MB', 'BAF', 'BAF_SE', 'DEPTH', 'DEPTH_SE', 'COPY_CHANGE', 'CELL_FRAC']] \
    .to_csv('WGS_500k.mCA.anon.csv', index=False)

In [ ]:
pd.value_counts(
    pd.value_counts(df['ID']) \
        .to_frame() \
        .reset_index() \
        .merge(df_age["ID"], how = 'outer') \
        .fillna(0) \
        .astype(int)['count']
) \
    .to_frame() \
    .reset_index(names='index') \
    .rename(columns={'index':'mCAs', 'count':'Individuals'}) \
    .sort_values('mCAs') \
    .assign(mCAs = lambda x: x['mCAs'].apply(lambda x: f"20+" if x >= 20  else x)) \
    .groupby('mCAs', as_index=False)['Individuals'].sum() \
    .to_csv('per_ind_mCA_counts.csv', index=False)